In [1]:
# Bibliotecas para trabalhar com os dados
import numpy as np
import pandas as pd
import pingouin as pg

# Seleção e avaliação dos modelos
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.model_selection import cross_validate, GridSearchCV

# Preparação dos dados
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

# Modelos preditivos
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor

# Métricas
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Gráficos
import plotly.express as px
import plotly.io as pio

#sistema
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 5)

pio.templates.default = 'plotly_dark'
px.defaults.height = 500
px.defaults.color_continuous_scale = px.colors.sequential.Viridis

In [2]:
PASTA_DADOS = Path(".")
if not (PASTA_DADOS / "transactions_train.csv").is_file():
    PASTA_DADOS = Path("H&M - Hackaton")

N_LINHAS = 1_000_000  # None para ler o arquivo completo.

df_transaction = pd.read_csv(
    PASTA_DADOS / "transactions_train.csv",
    index_col=False,
    dtype={"customer_id": "string", "article_id": "string"},
    nrows=N_LINHAS,
)

df_transaction['price'] = df_transaction['price'].astype(float).round(4)

df_transaction['t_dat'] = pd.to_datetime(df_transaction['t_dat'], format='%Y-%m-%d')

df_transaction

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.0508,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.0305,2
...,...,...,...,...,...
999998,2018-10-11,46193b292b8c4cbef8d2b75a75b92584c4fef588b07171...,0524825019,0.0339,2
999999,2018-10-11,461d77eece32034c78a9c1425f84b707bffed6d86368c0...,0461414008,0.0393,2


In [3]:

df_clientes = pd.read_csv(
    "customers.csv",
    index_col=False,
    dtype={"customer_id": "string"},
    nrows = N_LINHAS
)

display(df_clientes)

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
...,...,...,...,...,...,...,...
999998,ba9ade47e568d76b62f274b023fbc73acfba04e38c1082...,1.0,1.0,ACTIVE,Regularly,62.0,0c9f3dc39438e945c5c8fbafbbba0d67b3fd627f667084...
999999,ba9ae8434966611eb42e03e0bfca7eac3e7e1c16dbc210...,NaN,NaN,ACTIVE,NONE,32.0,caaebcb2559fb0b6148b762b2341fc9064936aa1baf4bc...


In [5]:
df_main = df_transaction.merge(
      df_clientes[['customer_id', 'club_member_status']]
    , on='customer_id'
    , how='left'
    , validate='many_to_one'
)

df_main

,t_dat,customer_id,article_id,price,sales_channel_id,club_member_status
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.0508,2,ACTIVE
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.0305,2,ACTIVE
...,...,...,...,...,...,...
999998,2018-10-11,46193b292b8c4cbef8d2b75a75b92584c4fef588b07171...,0524825019,0.0339,2,ACTIVE
999999,2018-10-11,461d77eece32034c78a9c1425f84b707bffed6d86368c0...,0461414008,0.0393,2,ACTIVE


In [6]:
df_articles = pd.read_csv(
    "articles.csv",
    index_col=False,
    dtype={"article_id": "string"},
    nrows=N_LINHAS
)

df_articles.drop(columns=['detail_desc'], inplace=True)

for column in ['prod_name', 'product_type_name', 'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'perceived_colour_value_name', 'perceived_colour_master_name', 'department_name', 'index_name', 'index_group_name', 'section_name', 'garment_group_name']:
    df_articles[column] = (
            df_articles[column]
            .astype("string")
            .dropna()
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", "_", regex=True)
    )

df_articles

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name
0,0108775015,108775,strap_top,253,vest_top,garment_upper_body,1010016,solid,9,black,4,dark,5,black,1676,jersey_basic,A,ladieswear,1,ladieswear,16,womens_everyday_basics,1002,jersey_basic
1,0108775044,108775,strap_top,253,vest_top,garment_upper_body,1010016,solid,10,white,3,light,9,white,1676,jersey_basic,A,ladieswear,1,ladieswear,16,womens_everyday_basics,1002,jersey_basic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105540,0957375001,957375,claire_hair_claw,72,hair_clip,accessories,1010016,solid,9,black,4,dark,5,black,3946,small_accessories,D,divided,2,divided,52,divided_accessories,1019,accessories
105541,0959461001,959461,lounge_dress,265,dress,garment_full_body,1010016,solid,11,off_white,1,dusty_light,9,white,1641,jersey,A,ladieswear,1,ladieswear,18,womens_trend,1005,jersey_fancy


In [7]:
df_main = df_main.merge(
      df_articles[['article_id', 'product_group_name']]
    , on='article_id'
    , how='left'
    , validate='many_to_one'
)

df_main

,t_dat,customer_id,article_id,price,sales_channel_id,club_member_status,product_group_name
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.0508,2,ACTIVE,underwear
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.0305,2,ACTIVE,underwear
...,...,...,...,...,...,...,...
999998,2018-10-11,46193b292b8c4cbef8d2b75a75b92584c4fef588b07171...,0524825019,0.0339,2,ACTIVE,garment_lower_body
999999,2018-10-11,461d77eece32034c78a9c1425f84b707bffed6d86368c0...,0461414008,0.0393,2,ACTIVE,shoes


In [15]:
with pd.option_context("display.max_rows", None):
    display(df_main.describe())
    df_main.info()


,t_dat,price,sales_channel_id
count,1000000,1000000.000000,1000000.000000
mean,2018-09-29 21:07:43.190400256,0.030177,1.662943
min,2018-09-20 00:00:00,0.000700,1.000000
25%,2018-09-25 00:00:00,0.016900,1.000000
50%,2018-09-29 00:00:00,0.025400,2.000000
75%,2018-10-05 00:00:00,0.033900,2.000000
max,2018-10-11 00:00:00,0.591500,2.000000
std,NaN,0.022915,0.472705


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 7 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   t_dat               1000000 non-null  datetime64[ns]
 1   customer_id         1000000 non-null  string        
 2   article_id          1000000 non-null  string        
 3   price               1000000 non-null  float64       
 4   sales_channel_id    1000000 non-null  int64         
 5   club_member_status  729572 non-null   object        
 6   product_group_name  1000000 non-null  string        
dtypes: datetime64[ns](1), float64(1), int64(1), object(1), string(3)
memory usage: 53.4+ MB


In [12]:
import plotly.express as px

resumo = (
    df_main["product_group_name"]
    .value_counts()
    .reset_index()
)

fig = px.bar(
    resumo,
    x="product_group_name",
    y="count",
    title="Quantidade de artigos vendidos por categoria",
    labels={
        "product_group_name": "Categoria",
        "count": "Quantidade vendida"
    }
)

fig.show()


compras_dia = (
    df_main.groupby("t_dat")
    .size()
    .reset_index(name="qtd_compras")
)

fig = px.line(
    compras_dia,
    x="t_dat",
    y="qtd_compras",
    title="Evolução diária das compras"
)

fig.show()